In [2]:
import json

# Open the JSON file for reading
with open('../data/ophthalmology_test_sets/BCSC.json', 'r') as file:
    # Parse the JSON file into a Python dictionary
    data = json.load(file)

# Now you can use the 'data' dictionary
print(data)


[{'query': 'Given a multiple choice question in the field of ophthalmology, select the correct answer from the four options.\n\nQuestion: A 30-year-old man was using an electric power saw on a piece of wood and hit a nail. He noticed a sharp pain in the left eye, with worsening vision over the next 2 days. He was not wearing goggles. He presents with visual acuity of 20/20 and redness and chemosis inferotemporally in his left eye 4 mm from the limbus. The pupil is round and reactive. Dilated examination is remarkable for a localized hemorrhage in the inferior vitreous, but no foreign body is visualized. What is the next appropriate step in management?\nA. computed tomography (CT) of the orbits\nB. close observation\nC. magnetic resonance imaging (MRI) of the orbits\nD. operative exploration', 'answer': 'A', 'id': '0'}, {'query': 'Given a multiple choice question in the field of ophthalmology, select the correct answer from the four options.\n\nQuestion: Aside from axial myopia, what is

In [18]:
def standardize_question(text):
    pattern = r"Question: (.*?)\n(A\..*?)(\nB\..*?)(\nC\..*?)(\nD\..*)"

    match = re.search(pattern, text, re.DOTALL)
    if match:
        question = match.group(1)
        choices = {
            "A": match.group(2).strip()[3:],  # Remove 'A. ' from the beginning
            "B": match.group(3).strip()[3:],  # Remove 'B. ' from the beginning
            "C": match.group(4).strip()[3:],  # Remove 'C. ' from the beginning
            "D": match.group(5).strip()[3:]   # Remove 'D. ' from the beginning
        }
        return question, choices
    else:
        print("[ERROR]: ", text)


def get_answer(cot, q):
    try:
        answer, _, _ = cot.answer(question=q['question'], options={"A": q['opa'], "B": q['opb'], "C": q['opc'], "D": q['opd']})
    except: 
        return None
    try:
        answer = json.loads(answer)
        return answer['answer_choice']
    except:
        list_of_string = re.findall(r'"(.*?)"', answer)
        if len(list_of_string) != 0:
            return list_of_string[-1]
        else:
            print('Output is not in json format!')
            print(answer)
            return None

In [12]:
import json
import sys
import os
# Temporarily add MedRAG repo
sys.path.append(os.path.abspath("../MedRAG"))
from src.medrag import MedRAG
from tqdm import tqdm
import re
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

def standardize_question(text):
    pattern = r"Question: (.*?)\n(A\..*?)(\nB\..*?)(\nC\..*?)(\nD\..*)"

    match = re.search(pattern, text, re.DOTALL)
    if match:
        question = match.group(1)
        choices = {
            "A": match.group(2).strip()[3:],  # Remove 'A. ' from the beginning
            "B": match.group(3).strip()[3:],  # Remove 'B. ' from the beginning
            "C": match.group(4).strip()[3:],  # Remove 'C. ' from the beginning
            "D": match.group(5).strip()[3:]   # Remove 'D. ' from the beginning
        }
        return question, choices
    else:
        print("[ERROR]: ", text)


def get_answer(cot, q, c):
    try:
        answer, _, _ = cot.answer(question=q, options={"A": c['A'], "B": c['B'], "C": c['C'], "D": c['D']})
    except: 
        return None
    try:
        answer = json.loads(answer)
        return answer['answer_choice']
    except:
        if isinstance(answer, str):
            list_of_string = re.findall(r'"(.*?)"', answer)
            if len(list_of_string) != 0:
                return list_of_string[-1]
            print('Output is not in json format or answer is not string!')
            print(answer)
            return None
        
            

model_names = ["OpenAI/gpt-35-turbo-16k", "OpenAI/gpt-4o"]
retriever_names = ["MedCPT"]
# corpus_names = ["PubMed", "StatPearls", "Textbooks", "Wikipedia", "MedCorp"]
corpus_names = ["Textbooks"]

with open('../data/ophthalmology_test_sets/BCSC.json', 'r') as file:
    data = json.load(file)

data = data[:2]

output_all = ""
for model_name_ in model_names:
    for retriever_name_ in retriever_names:
        for corpus_name_ in corpus_names:
            output_name_ = "BCSC+" + model_name_ + "+" + retriever_name_ + "+" + corpus_name_
            print('---', output_name_, '---')
            cot = MedRAG(llm_name=model_name_, rag=True, retriever_name=retriever_name_, corpus_name=corpus_name_)
            correct, incorrect = 0, 0
            unknown = 0
            true_labels = []
            predicted_labels = []
            for i, question in tqdm(enumerate(data)):
                if output_name_ in data[i].keys():
                    continue
                else:
                    q, c = standardize_question(question['query'])
                    a = get_answer(cot, q, c)
                    # mapper = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
                    true_labels.append(question['answer'])
                    try:
                        ans = (a[0] == question['answer']) # True for correct answer, False otherwise
                        predicted_labels.append(a[0])
                        if ans:
                            correct += 1
                        else:
                            incorrect += 1
                    except:
                        predicted_labels.append(0)
                        unknown += 1
                        print(question)
                        print(a)
                    data[i][output_name_] = a
            label_encoder = LabelEncoder()
            label_encoder.fit(true_labels+predicted_labels)
            true_encoded = label_encoder.transform(true_labels)
            predicted_encoded = label_encoder.transform(predicted_labels)
            macro_f1 = f1_score(true_encoded, predicted_encoded, average='macro')
            output = f"{output_name_}. # of Correct Answer: {correct}, # of Incorrect Answer: {incorrect}, # of Not-identified Pattern: {unknown}. Macro-f1: {macro_f1}."
            print(output)
            output_all += output + "\n"

with open('../data/results/BSCS_result.json', 'w') as file:
    json.dump(people, file)
with open("../data/results/BSCS_result.txt", "w") as file:
    file.write(output_all)

--- BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+Textbooks ---
No sentence-transformers model found with name /home/zc347/.cache/torch/sentence_transformers/ncbi_MedCPT-Query-Encoder. Creating a new one with CLS pooling.


0it [00:00, ?it/s]

False
aa


1it [00:03,  4.00s/it]

False
aa


...

False
aa


1it [00:07,  7.66s/it]

False
aa


2it [00:13,  6.82s/it]

BCSC+OpenAI/gpt-4o+MedCPT+Textbooks. # of Correct Answer: 1, # of Incorrect Answer: 1, # of Not-identified Pattern: 0. Macro-f1: 0.3333333333333333.


In [ ]:
# model_name_ = "OpenAI/gpt-4o"
model_name_ = "OpenAI/gpt-35-turbo-16k"
retriever_name_ = "MedCPT"
# corpus_name_ = "MedCorp"
corpus_name_ = "Textbooks"
output_name_ = "MedMCQA_train_top1k+" + model_name_ + "+" + retriever_name_ + "+" + corpus_name_

data_train = []
with open("../MedMCQA/train.json", "r") as file:
    for line in file:
        question = json.loads(line)
        data_train.append(question)

data_train_top1k = data_train[:3]

def get_answer(cot, q):
    try:
        answer, _, _ = cot.answer(question=q['question'], options={"A": q['opa'], "B": q['opb'], "C": q['opc'], "D": q['opd']})
    except: 
        return None
    try:
        answer = json.loads(answer)
        return answer['answer_choice']
    except:
        list_of_string = re.findall(r'"(.*?)"', answer)
        if len(list_of_string) != 0:
            return list_of_string[-1]
        else:
            print('Output is not in json format!')
            print(answer)
            return None


cot = MedRAG(llm_name=model_name_, rag=True, retriever_name=retriever_name_, corpus_name=corpus_name_)
correct, incorrect = 0, 0
unknown = 0
true_labels = []
predicted_labels = []
for question in tqdm(data_train_top1k):
    a = get_answer(cot, question)
    mapper = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
    true_labels.append(question['cop'])
    try:
        ans = (mapper[a[0]] == question['cop']) # True for correct answer, False otherwise
        predicted_labels.append(mapper[a[0]])
        if ans:
            correct += 1
        else:
            incorrect += 1
    except:
        predicted_labels.append(0)
        unknown += 1
        print(question)
        print(a)


label_encoder = LabelEncoder()
label_encoder.fit(true_labels+predicted_labels)
true_encoded = label_encoder.transform(true_labels)
predicted_encoded = label_encoder.transform(predicted_labels)
macro_f1 = f1_score(true_encoded, predicted_encoded, average='macro')

output = f"{output_name_}. # of Correct Answer: {correct}, # of Incorrect Answer: {incorrect}, # of Not-identified Pattern: {unknown}. Macro-f1: {macro_f1}."
print(output)
file_path = output_name_ + ".txt"
file_path = file_path.replace("/", "-")
with open(file_path, "w") as file:
    file.write(output)


In [ ]:
import json
import sys
import os
# Temporarily add MedRAG repo
sys.path.append(os.path.abspath("../MedRAG"))
from src.medrag import MedRAG
from tqdm import tqdm
import re
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

def standardize_question(text):
    pattern = r"Question: (.*?)\n(A\..*?)(\nB\..*?)(\nC\..*?)(\nD\..*)"

    match = re.search(pattern, text, re.DOTALL)
    if match:
        question = match.group(1)
        choices = {
            "A": match.group(2).strip()[3:],  # Remove 'A. ' from the beginning
            "B": match.group(3).strip()[3:],  # Remove 'B. ' from the beginning
            "C": match.group(4).strip()[3:],  # Remove 'C. ' from the beginning
            "D": match.group(5).strip()[3:]   # Remove 'D. ' from the beginning
        }
        return question, choices
    else:
        print("[ERROR]: ", text)


def get_answer(cot, q, c):
    try:
        answer, _, _ = cot.answer(question=q, options={"A": c['A'], "B": c['B'], "C": c['C'], "D": c['D']})
    except: 
        return None
    try:
        answer = json.loads(answer)
        return answer['answer_choice']
    except:
        if isinstance(answer, str):
            list_of_string = re.findall(r'"(.*?)"', answer)
            if len(list_of_string) != 0:
                return list_of_string[-1]
            print('Output is not in json format or answer is not string!')
            print(answer)
            return None
        
            

model_names = ["OpenAI/gpt-35-turbo-16k", "OpenAI/gpt-4o"]
retriever_names = ["MedCPT"]
# corpus_names = ["PubMed", "StatPearls", "Textbooks", "Wikipedia", "MedCorp"]
corpus_names = ["Textbooks", "PubMed"]

with open('../data/ophthalmology_test_sets/BCSC.json', 'r') as file:
    data = json.load(file)

data = data[:2]

output_all = ""
for model_name_ in model_names:
    for retriever_name_ in retriever_names:
        for corpus_name_ in corpus_names:
            output_name_ = "BCSC+" + model_name_ + "+" + retriever_name_ + "+" + corpus_name_
            print('---', output_name_, '---')
            cot = MedRAG(llm_name=model_name_, rag=True, retriever_name=retriever_name_, corpus_name=corpus_name_)
            correct, incorrect = 0, 0
            unknown = 0
            true_labels = []
            predicted_labels = []
            for question in tqdm(data):
                q, c = standardize_question(question['query'])
                a = get_answer(cot, q, c)
                print(a)
                # mapper = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
                true_labels.append(question['answer'])
                print("answer:", a[0], "; true answer:", question['answer'])
                try:
                    print("[TRY BLOCK]", a[0] == question['answer'])
                    ans = (a[0] == question['answer']) # True for correct answer, False otherwise
                    predicted_labels.append(a[0])
                    if ans:
                        correct += 1
                    else:
                        incorrect += 1
                except:
                    predicted_labels.append(0)
                    unknown += 1
                    print(question)
                    print(a)
            label_encoder = LabelEncoder()
            label_encoder.fit(true_labels+predicted_labels)
            true_encoded = label_encoder.transform(true_labels)
            predicted_encoded = label_encoder.transform(predicted_labels)
            macro_f1 = f1_score(true_encoded, predicted_encoded, average='macro')
            output = f"{output_name_}. # of Correct Answer: {correct}, # of Incorrect Answer: {incorrect}, # of Not-identified Pattern: {unknown}. Macro-f1: {macro_f1}."
            print(output)
            output_all += output + "\n"
            
with open("BSCS_result.txt", "w") as file:
    file.write(output_all)

/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+Textbooks ---
No sentence-transformers model found with name /home/zc347/.cache/torch/sentence_transformers/ncbi_MedCPT-Query-Encoder. Creating a new one with CLS pooling.


/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
 50%|█████     | 1/2 [00:04<00:04,  4.91s/it]

answer: A ; true answer: A
[TRY BLOCK] True


100%|██████████| 2/2 [00:12<00:00,  6.02s/it]

answer: A ; true answer: C
[TRY BLOCK] False
BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+Textbooks. # of Correct Answer: 1, # of Incorrect Answer: 1, # of Not-identified Pattern: 0. Macro-f1: 0.3333333333333333.
--- BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+PubMed ---


In [10]:
from pyserini.search.lucene import LuceneSearcher